In [ ]:
import torch
import torch.nn as nn
from typing import Optional


class ValueNet(nn.Module):
    def __init__(self, obs_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        # returns shape: (batch,) as float
        v = self.net(obs).squeeze(-1)
        return v


class TDLambda:
    """
    Semi-gradient TD(lambda) for state-value function approximation.

    Update:
        delta = r + gamma * (1-done) * V(s_next) - V(s)
        e = gamma * lambda * e + grad_theta V(s)
        theta += alpha * delta * e
    """
    def __init__(
        self,
        value_net: nn.Module,
        alpha: float = 1e-3,
        gamma: float = 0.99,
        lam: float = 0.95,
        device: Optional[torch.device] = None,
    ):
        self.v = value_net
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.lam = float(lam)
        self.device = device if device is not None else torch.device("cpu")
        self.v.to(self.device)

        # eligibility traces: one tensor per parameter, same shape
        self.traces = [torch.zeros_like(p, device=self.device) for p in self.v.parameters()]

    @torch.no_grad()
    def reset_traces(self):
        for e in self.traces:
            e.zero_()

    def step(self, s, r, s_next, done: bool):
        """
        Perform one TD(lambda) update from transition (s, r, s_next, done).

        s, s_next: array-like or torch tensors of shape (obs_dim,)
        r: float
        done: bool
        """
        # Convert to tensors
        if not torch.is_tensor(s):
            s = torch.tensor(s, dtype=torch.float32, device=self.device)
        else:
            s = s.to(self.device).float()

        if not torch.is_tensor(s_next):
            s_next = torch.tensor(s_next, dtype=torch.float32, device=self.device)
        else:
            s_next = s_next.to(self.device).float()

        r_t = torch.tensor(r, dtype=torch.float32, device=self.device)
        done_t = torch.tensor(float(done), dtype=torch.float32, device=self.device)

        # Ensure grads are clean
        self.v.zero_grad(set_to_none=True)

        # Forward for current state WITH grad
        v_s = self.v(s)

        # Forward for next state WITHOUT grad in TD target (semi-gradient)
        with torch.no_grad():
            v_next = self.v(s_next)
            target = r_t + self.gamma * (1.0 - done_t) * v_next

        # TD error (scalar)
        delta = target - v_s  # keeps grad path through v_s only

        # Compute grad of V(s) wrt params: ∇θ V(s)
        v_s.backward()  # grads now in p.grad

        # Update traces and parameters
        with torch.no_grad():
            decay = self.gamma * self.lam
            for i, p in enumerate(self.v.parameters()):
                if p.grad is None:
                    continue
                # accumulating trace
                self.traces[i].mul_(decay).add_(p.grad)

                # theta += alpha * delta * e
                p.add_(self.alpha * delta * self.traces[i])

        # If episode ended, typically reset traces
        if done:
            self.reset_traces()

        # Return useful diagnostics
        return {
            "v_s": float(v_s.detach().cpu()),
            "target": float(target.detach().cpu()),
            "delta": float(delta.detach().cpu()),
        }


In [ ]:
obs_dim = 4
vnet = ValueNet(obs_dim)
agent = TDLambda(vnet, alpha=1e-3, gamma=0.99, lam=0.95)

agent.reset_traces()

# example transition
s = [0.1, 0.0, -0.2, 0.3]
r = 1.0
s_next = [0.12, 0.01, -0.18, 0.28]
done = False

info = agent.step(s, r, s_next, done)
print(info)
